# Multimodal search on your laptop

Photos, audio and text in one search, with EmbeddingGemma 2.

1. One model, three parts
2. Load everything once and see each part's size
3. Turn photos, audio and text into embeddings, one type at a time
4. Search all of them with the text part only
5. Cut the embeddings to 128 numbers

In [1]:
import os, gc, subprocess
os.environ.setdefault("HF_HOME", "D:/hf-cache")

import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

MODEL_ID = "google/embeddinggemma-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 or float32 only: in float16, long documents silently turn into NaN
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
FILES = Path("demo_files")
print(DEVICE, DTYPE)

d:\atliq_work\jev\next-use-cases\killtest\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda torch.bfloat16


## 1. One model, three parts

A text part (always there), an image part and an audio part. Choose which ones to load.

In [2]:
ENCODERS = {
    "text only":    {"vision_config": None, "audio_config": None},
    "text + image": {"audio_config": None},
    "text + audio": {"vision_config": None},
    "everything":   {},
}

def load(encoders: str, device: str = DEVICE, dtype=DTYPE) -> SentenceTransformer:
    return SentenceTransformer(
        MODEL_ID,
        device=device,
        model_kwargs={"torch_dtype": dtype},
        config_kwargs=ENCODERS[encoders],
    )

def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 2. Load everything once, and see how big each part is

In [3]:
model = load("everything")

parts = {"text": 0, "image": 0, "audio": 0}
for name, p in model.named_parameters():
    part = "image" if "vision" in name else "audio" if "audio" in name else "text"
    parts[part] += p.numel()
for part, n in parts.items():
    print(f"{part:<6}{n / 1e6:>5.0f}M parameters")
print(f"total {sum(parts.values()) / 1e6:>5.0f}M")

Loading weights: 100%|██████████| 1376/1376 [00:02<00:00, 557.18it/s]


text    271M parameters
image   168M parameters
audio   306M parameters
total   744M


## 3. Index once, with the full model

Each type of file becomes embeddings, one type at a time. For every embedding we also keep where it came from.

### Photos: one photo, one embedding

In [4]:
vectors, where = [], []

for path in sorted((FILES / "images").iterdir()):
    v = model.encode({"image": str(path)}, normalize_embeddings=True)
    vectors.append(v)
    where.append({"file": path.name, "start": 0})
    print(f"{path.name:<22} -> {v.shape}")

colour_powders.jpg     -> (768,)
junction.jpg           -> (768,)


### Audio: cut into 30-second pieces, one embedding per piece

So a search can find the right moment in a long recording.

In [5]:
def length(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(path)],
                         capture_output=True, text=True).stdout
    return float(out)

def cut(path, start, seconds=30):
    out = f"piece_{path.stem}_{start}.wav"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-ss", str(start), "-t", str(seconds), "-i", str(path),
                    "-ac", "1", "-ar", "16000", out], check=True)   # the model expects mono, 16 kHz
    return out

for path in sorted((FILES / "audio").iterdir()):
    starts = range(0, int(length(path)), 30)
    for s in starts:
        vectors.append(model.encode({"audio": cut(path, s)}, normalize_embeddings=True))
        where.append({"file": path.name, "start": s})
    print(f"{path.name:<22} {length(path):5.0f} s -> {len(starts)} piece(s)")

armstrong_1969.mp3        24 s -> 1 piece(s)
roosevelt_1933.mp3        47 s -> 2 piece(s)


### Text: with the Document prompt

The model was trained with a short label in front of text: one for documents, one for search queries.

In [6]:
for path in sorted((FILES / "text").iterdir()):
    v = model.encode(path.read_text(encoding="utf-8"), prompt_name="Document", normalize_embeddings=True)
    vectors.append(v)
    where.append({"file": path.name, "start": 0})
    print(f"{path.name:<22} -> {v.shape}")

grocery_note.txt       -> (768,)
vet_note.txt           -> (768,)


All of them, in one table of numbers:

In [7]:
vectors = np.array(vectors)
del model
release()
print(vectors.shape)

(7, 768)


## 4. Search with the text part only

One embedding for the query, compare it with every stored embedding, keep the best piece of each file.

In [8]:
searcher = load("text only")
print(f"text only: {sum(p.numel() for p in searcher.parameters()) / 1e6:.0f}M parameters")

def search(query, k=3, dim=768):
    q = searcher.encode(query, prompt_name="SearchQuery")[:dim]
    v = vectors[:, :dim] / np.linalg.norm(vectors[:, :dim], axis=1, keepdims=True)   # normalize (again, after a cut)
    scores = v @ (q / np.linalg.norm(q))                           # normalized, so this is cosine similarity
    best = {}
    for i in np.argsort(-scores):
        best.setdefault(where[i]["file"], i)                       # keep the best piece of each file
    print(f"'{query}'")
    for i in list(best.values())[:k]:
        print(f"   {scores[i]:.2f}  {where[i]['file']}  at {where[i]['start']}s")

for q in ["the only thing we have to fear is fear itself", "a sign that says wear helmet",
          "colourful powder in a shop", "things to buy at the shop", "a dog's doctor visit"]:
    search(q)

Loading weights: 100%|██████████| 413/413 [00:00<00:00, 1052.61it/s]


text only: 271M parameters
'the only thing we have to fear is fear itself'
   0.71  roosevelt_1933.mp3  at 0s
   0.61  armstrong_1969.mp3  at 0s
   0.59  vet_note.txt  at 0s
'a sign that says wear helmet'
   0.68  junction.jpg  at 0s
   0.61  vet_note.txt  at 0s
   0.58  roosevelt_1933.mp3  at 0s
'colourful powder in a shop'
   0.79  colour_powders.jpg  at 0s
   0.64  grocery_note.txt  at 0s
   0.59  vet_note.txt  at 0s
'things to buy at the shop'
   0.67  grocery_note.txt  at 0s
   0.65  colour_powders.jpg  at 0s
   0.63  vet_note.txt  at 0s
'a dog's doctor visit'
   0.72  vet_note.txt  at 0s
   0.57  roosevelt_1933.mp3  at 0s
   0.56  armstrong_1969.mp3  at 0s


## 5. Cut every embedding to 128 numbers (Matryoshka)

Six times smaller. Do the top results change?

In [9]:
for q in ["the only thing we have to fear is fear itself", "a sign that says wear helmet", "things to buy at the shop"]:
    search(q, k=1, dim=128)

'the only thing we have to fear is fear itself'
   0.82  roosevelt_1933.mp3  at 0s
'a sign that says wear helmet'
   0.81  junction.jpg  at 0s
'things to buy at the shop'
   0.81  grocery_note.txt  at 0s
